# Urban Planner GRPO — T4-optimized

Unsloth + TRL GRPO for the OpenEnv Urban Planner. Tuned for **Colab T4 (15.6 GB VRAM)**.

**Why this rewrite is different from the previous notebook**
- `beta=0.0` ⇒ TRL skips the reference model ⇒ ~3.5 GB freed ⇒ no OOM.
- `MAX_SEQ_LENGTH=640` (was 1024) ⇒ ~35% less activation memory + faster forward.
- Reward range widened to **[-1, +1]** with a **rubric-Δ** core signal so GRPO group std no longer collapses to 0.
- One shared env is reused across reward calls (was rebuilt every call) — ~3× faster reward eval.
- LoRA `r=16, α=32` (was r=8) for more learning capacity, still T4-safe.
- Generation `temperature=1.0, top_p=0.95, top_k=50` ⇒ in-group diversity ⇒ usable advantages.
- `max_grad_norm=1.0` (was 0.05 — that was silently zeroing gradients).

## 1) Install dependencies

In [ ]:
%pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install -q "trl>=0.11" "transformers>=4.44" "peft>=0.12" accelerate datasets bitsandbytes matplotlib numpy pydantic
%pip install -q fastmcp openenv-core fastapi uvicorn httpx

## 2) Allocator + GPU sanity check

`expandable_segments:True` recovers ~0.5–1 GB of headroom on T4 by letting the allocator reuse fragmented blocks.

In [ ]:
import os
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

import torch, gc
torch.cuda.empty_cache(); gc.collect()

assert torch.cuda.is_available(), 'CUDA not available — switch Colab runtime to GPU.'
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {torch.cuda.get_device_name(0)}  |  VRAM: {vram:.1f} GB')
assert vram >= 12.0, f'Need >=12 GB VRAM, got {vram:.1f} GB.'

## 3) Clone the env repo and put it on `sys.path`

Replace `REPO_URL` with your fork if you've changed the env.

In [ ]:
import subprocess, sys, os
REPO_URL = 'https://huggingface.co/spaces/kanishjn8/openenv-urban-planner'
REPO_DIR = 'repo'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
sys.path.insert(0, os.path.abspath(REPO_DIR))

from server.urban_planner_environment import UrbanPlannerEnvironment
from server.rubric import urban_planner_rubric
from models import EpisodeConfig
print('Env imports OK')

## 4) Hyperparameters

Each value is justified at the bottom of this notebook.

In [ ]:
from pathlib import Path

MODEL_NAME            = 'unsloth/Qwen2.5-3B-Instruct'
MAX_SEQ_LENGTH        = 640        # prompt 512 + completion 128
MAX_PROMPT_LEN        = 512
MAX_NEW_TOKENS        = 128

LORA_R                = 16
LORA_ALPHA            = 32
LORA_DROPOUT          = 0.0

NUM_GENERATIONS       = 4
PER_DEVICE_BATCH      = 1
GRAD_ACCUM            = 4          # effective batch = 4 prompts × 4 gens = 16
MAX_STEPS             = 200

LEARNING_RATE         = 5e-6
WARMUP_RATIO          = 0.05
WEIGHT_DECAY          = 0.01
MAX_GRAD_NORM         = 1.0

BETA                  = 0.0        # 0 ⇒ no reference model loaded → saves ~3.5 GB
EPSILON               = 0.2
TEMPERATURE           = 1.0
TOP_P                 = 0.95
TOP_K                 = 50
REPETITION_PENALTY    = 1.1

N_EPISODES            = 200
STEPS_PER_EP          = 3
STARTING_BUDGET       = 10_000
RNG_SEED              = 3407

OUTPUT_DIR            = './urban_planner_grpo_t4'
SAVE_DIR              = './urban_planner_grpo_t4_final'
PLOTS_DIR             = Path('assets/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print('Config loaded.')

## 5) Load model with Unsloth (4-bit QLoRA)

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,        # bf16 if supported else fp16
    load_in_4bit   = True,
)
print('Model loaded:', model.config.name_or_path)

## 6) Add LoRA adapters

`use_gradient_checkpointing='unsloth'` saves ~30% activation memory vs the stock GC.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r              = LORA_R,
    lora_alpha     = LORA_ALPHA,
    lora_dropout   = LORA_DROPOUT,
    bias           = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state   = RNG_SEED,
    target_modules = [
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
)
model.print_trainable_parameters()

## 7) System prompt + observation formatter

The system prompt is intentionally **short and schema-strict** so the model spends its capacity learning content, not formatting.

In [ ]:
VALID_TOOLS = {
    'get_city_state', 'get_district_report', 'place_zone',
    'place_infrastructure', 'allocate_budget', 'query_residents',
    'query_traffic_model', 'advance_season', 'get_event_log',
    'get_budget_report',
}

SYSTEM_PROMPT = (
    'You are an urban planner. Output exactly ONE JSON object describing one tool call.\n'
    'Schema: {"name":"<tool>","arguments":{...}}\n'
    'Allowed tools:\n'
    ' - get_city_state(region)\n'
    ' - get_district_report(district_id)\n'
    ' - place_zone(x,y,zone_type,density)  zone_type ∈ residential|commercial|industrial|green|transit\n'
    ' - place_infrastructure(x,y,infra_type)  infra_type ∈ road|metro|hospital|school|flood_barrier\n'
    ' - allocate_budget(category,amount)  category ∈ maintenance|expansion|emergency\n'
    ' - query_residents(district_id)\n'
    ' - query_traffic_model(origin,destination)\n'
    ' - advance_season()\n'
    ' - get_event_log(last_n)\n'
    ' - get_budget_report()\n'
    'Heuristics: roads near residential, schools before density, no industrial next to residential, '
    'keep $500+ emergency fund, advance season periodically.\n'
    'OUTPUT ONLY THE JSON. No prose, no markdown, no <tool_call> tags.\n'
    'Example: {"name":"place_infrastructure","arguments":{"x":7,"y":8,"infra_type":"road"}}'
)

def observation_to_text(obs) -> str:
    parts = [f'Season {obs.season} | Budget ${obs.budget_remaining}']
    if obs.event_log:
        parts.append('Events: ' + ' | '.join(obs.event_log[-2:]))
    if obs.policy_constraints:
        parts.append('Policy: ' + '; '.join(obs.policy_constraints[:2]))
    if obs.planning_log:
        log_lines = []
        for e in obs.planning_log[-3:]:
            log_lines.append(f'S{e.season} {e.action_summary[:60]} dr={e.reward_delta:+.2f}')
        parts.append('Log: ' + ' || '.join(log_lines))
    parts.append(f'Tool result: {(obs.tool_result or "")[:200]}')
    return '\n'.join(parts)

def format_prompt(obs_text: str) -> str:
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': obs_text + '\nReply with one JSON tool call.'},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

## 8) Build the dataset

200 seeds × ≤3 steps ≈ 600 prompts. Each row carries `prompt`, `seed`, and `history` so the reward function can replay the prefix and score *this* completion's effect.

In [ ]:
import random, json
from datasets import Dataset

random.seed(RNG_SEED)
SEEDS = random.sample(range(50_000), N_EPISODES)

def random_action(rng):
    x, y = rng.randint(0, 15), rng.randint(0, 15)
    return rng.choice([
        {'tool_name': 'get_city_state',       'arguments': {'region': 'all'}},
        {'tool_name': 'get_district_report',  'arguments': {'district_id': rng.randint(0, 15)}},
        {'tool_name': 'place_zone',           'arguments': {'x': x, 'y': y,
            'zone_type': rng.choice(['residential', 'commercial', 'green']),
            'density': rng.randint(1, 2)}},
        {'tool_name': 'place_infrastructure', 'arguments': {'x': x, 'y': y,
            'infra_type': rng.choice(['road', 'school', 'flood_barrier'])}},
        {'tool_name': 'query_residents',      'arguments': {'district_id': rng.randint(0, 15)}},
        {'tool_name': 'get_budget_report',    'arguments': {}},
        {'tool_name': 'advance_season',       'arguments': {}},
    ])

def _build_rows_for_seed(seed, env):
    rng = random.Random(seed)
    obs = env.reset(EpisodeConfig(seed=seed, starting_budget=STARTING_BUDGET))
    history, rows = [], []
    for _ in range(STEPS_PER_EP):
        if obs.done:
            break
        rows.append({
            'prompt':  format_prompt(observation_to_text(obs)),
            'seed':    seed,
            'history': json.dumps(history),
        })
        a = random_action(rng)
        obs = env.step(a)
        history.append(a)
        history = history[-6:]
    return rows

_dataset_env = UrbanPlannerEnvironment()
all_rows = []
for s in SEEDS:
    all_rows.extend(_build_rows_for_seed(s, _dataset_env))
random.shuffle(all_rows)
train_dataset = Dataset.from_list(all_rows)
print(f'Dataset: {len(train_dataset)} rows from {N_EPISODES} seeds × ≤{STEPS_PER_EP} steps')
print('Sample prompt (first 400 chars):\n', train_dataset[0]['prompt'][:400])

## 9) Tool-call parser

Tolerates messy completions (Qwen sometimes emits `<tool_call>…</tool_call>`, single-quoted JSON, trailing commas) but is **strict** about the final shape. `None` ⇒ parse failure ⇒ -1.0 reward.

In [ ]:
import re, json

_TOOL_WRAPPED_RE = re.compile(r'<tool_call>\s*(\{.*?\})\s*</tool_call>', re.DOTALL)
_FIRST_JSON_RE   = re.compile(r'\{[\s\S]*\}')

def _normalize_text(completion):
    if isinstance(completion, str):
        return completion
    if isinstance(completion, dict):
        for k in ('content', 'text', 'completion', 'generated_text'):
            v = completion.get(k)
            if isinstance(v, str):
                return v
        return json.dumps(completion)
    if isinstance(completion, list):
        return '\n'.join(_normalize_text(x) for x in completion)
    return str(completion)

def _validate(payload):
    if not isinstance(payload, dict):
        return None
    name = payload.get('name') or payload.get('tool_name') or ''
    args = payload.get('arguments') or payload.get('args') or {}
    if name not in VALID_TOOLS or not isinstance(args, dict):
        return None
    return name, args

def _repair_json(s):
    s = s.strip().replace('\n', ' ').replace('\t', ' ')
    s = re.sub(r'\s+', ' ', s)
    s = s.replace("'", '"')
    s = re.sub(r',\s*\}', '}', s)
    s = re.sub(r',\s*\]', ']', s)
    return s

def parse_tool_call(completion):
    text = _normalize_text(completion)
    m = _TOOL_WRAPPED_RE.search(text)
    if m:
        try:
            r = _validate(json.loads(m.group(1)))
            if r is not None: return r
        except Exception:
            pass
    m = _FIRST_JSON_RE.search(text)
    if m:
        for cand in (m.group(0), _repair_json(m.group(0))):
            try:
                r = _validate(json.loads(cand))
                if r is not None: return r
            except Exception:
                pass
    return None

for c in [
    '{"name":"place_zone","arguments":{"x":3,"y":4,"zone_type":"residential","density":2}}',
    '<tool_call>{"name":"advance_season","arguments":{}}</tool_call>',
    'I think we should plan carefully.',
]:
    print(parse_tool_call(c))

## 10) Reward function

Wide-range, rubric-Δ driven. Designed to keep `reward_std` healthy and prevent the plateau the previous run hit.

| Component | Value |
|---|---|
| parse fail | **-1.00** |
| tool error (insufficient budget, invalid arg, etc.) | -0.40 |
| valid action base bonus | +0.15 |
| `4 × (rubric_after − rubric_before)` | main learning signal |
| `0.5 × env.reward` | env's shaped/rubric reward |
| repeated info tool last step | -0.05 |

Final reward is clipped to `[-1, +1]`.

In [ ]:
import numpy as np

PARSE_FAIL_REWARD        = -1.0
TOOL_ERROR_REWARD        = -0.4
VALID_ACTION_BONUS       = 0.15
RUBRIC_DELTA_WEIGHT      = 4.0
ENV_REWARD_WEIGHT        = 0.5
INFO_TOOL_REPEAT_PENALTY = 0.05
REWARD_LO, REWARD_HI     = -1.0, 1.0

_REWARD_ENV = None  # reuse one env across reward calls (no FastMCP rebuild)
def _get_reward_env():
    global _REWARD_ENV
    if _REWARD_ENV is None:
        _REWARD_ENV = UrbanPlannerEnvironment()
    return _REWARD_ENV

def _is_tool_error(tool_result):
    h = (tool_result or '').strip().lower()[:160]
    return (h.startswith('error') or '"error"' in h
            or 'insufficient budget' in h or 'already exists' in h
            or h.startswith('invalid'))

def reward_fn(completions, seed=None, history=None, **kwargs):
    if seed is None:
        seed = [42] * len(completions)
    if history is None:
        history = ['[]'] * len(completions)
    env = _get_reward_env()
    rewards, n_parse_fail, n_tool_err = [], 0, 0
    for completion, s, hist_json in zip(completions, seed, history):
        parsed = parse_tool_call(completion)
        if parsed is None:
            rewards.append(PARSE_FAIL_REWARD); n_parse_fail += 1; continue
        tool_name, arguments = parsed
        try:
            prior_actions = json.loads(hist_json) if hist_json else []
        except Exception:
            prior_actions = []
        try:
            env.reset(EpisodeConfig(seed=int(s), starting_budget=STARTING_BUDGET))
            for a in prior_actions[-3:]:
                try: env.step(a)
                except Exception: break
            score_before, _ = urban_planner_rubric.score(env._sim)
            obs = env.step({'tool_name': tool_name, 'arguments': arguments})
        except Exception:
            rewards.append(TOOL_ERROR_REWARD); n_tool_err += 1; continue
        if _is_tool_error(getattr(obs, 'tool_result', '')):
            rewards.append(TOOL_ERROR_REWARD); n_tool_err += 1; continue
        score_after, _ = urban_planner_rubric.score(env._sim)
        delta = score_after - score_before
        env_r = float(getattr(obs, 'reward', 0.0))
        r = (VALID_ACTION_BONUS
             + RUBRIC_DELTA_WEIGHT * delta
             + ENV_REWARD_WEIGHT * env_r)
        if prior_actions:
            last_tool = prior_actions[-1].get('tool_name')
            if (last_tool == tool_name and tool_name in (
                'get_city_state', 'get_budget_report', 'query_residents',
                'get_district_report', 'get_event_log')):
                r -= INFO_TOOL_REPEAT_PENALTY
        rewards.append(float(np.clip(r, REWARD_LO, REWARD_HI)))

    arr = np.asarray(rewards, dtype=float)
    if len(arr) >= 3:
        msg = (f'[reward] n={len(arr)} mean={arr.mean():+.3f} std={arr.std():.3f} '
               f'min={arr.min():+.2f} max={arr.max():+.2f} '
               f'parse_fail={n_parse_fail} tool_err={n_tool_err}')
        if arr.std() < 0.05:
            msg = '[WARN low variance] ' + msg
        print(msg)
    return rewards

# Smoke test
_smoke = reward_fn([
    '{"name":"place_infrastructure","arguments":{"x":7,"y":8,"infra_type":"road"}}',
    '{"name":"place_zone","arguments":{"x":3,"y":4,"zone_type":"residential","density":2}}',
    'I will think about budget and act',
    '{"name":"advance_season","arguments":{}}',
], seed=[42]*4, history=['[]']*4)
print('Smoke rewards:', [round(r, 3) for r in _smoke])

## 11) GRPO config

**`beta=0.0` is the single most important VRAM saving** — it tells TRL to skip loading the frozen reference model (which costs ~3.5 GB on a 3B in 4-bit). We rely on PPO-style epsilon clipping for stability.

Unsupported keys for the installed TRL version are silently dropped via `inspect`.

In [ ]:
import inspect
from trl import GRPOConfig

requested = dict(
    output_dir                  = OUTPUT_DIR,
    overwrite_output_dir        = False,
    per_device_train_batch_size = PER_DEVICE_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    max_grad_norm               = MAX_GRAD_NORM,
    num_generations             = NUM_GENERATIONS,
    beta                        = BETA,
    epsilon                     = EPSILON,
    max_steps                   = MAX_STEPS,
    logging_steps               = 2,
    save_steps                  = 50,
    save_total_limit            = 2,
    bf16                        = is_bfloat16_supported(),
    fp16                        = not is_bfloat16_supported(),
    report_to                   = 'none',
    seed                        = RNG_SEED,
    dataloader_num_workers      = 0,
    remove_unused_columns       = False,
    max_prompt_length           = MAX_PROMPT_LEN,
    max_completion_length       = MAX_NEW_TOKENS,
    temperature                 = TEMPERATURE,
    top_p                       = TOP_P,
    top_k                       = TOP_K,
    repetition_penalty          = REPETITION_PENALTY,
    use_vllm                    = False,
)
allowed = set(inspect.signature(GRPOConfig.__init__).parameters)
filtered = {k: v for k, v in requested.items() if k in allowed}
skipped  = sorted(set(requested) - set(filtered))
if skipped:
    print('Skipped unsupported GRPOConfig args:', skipped)
grpo_config = GRPOConfig(**filtered)
print(f'GRPO ready: beta={grpo_config.beta} num_generations={grpo_config.num_generations} max_steps={grpo_config.max_steps}')

## 12) Build the trainer and train

Auto-resumes from `OUTPUT_DIR/checkpoint-*` if present (helpful for Colab disconnects). Saves every 50 steps and keeps the last 2 checkpoints.

In [ ]:
from trl import GRPOTrainer
from transformers import GenerationConfig

_sig = inspect.signature(GRPOTrainer.__init__).parameters
_common = dict(model=model, reward_funcs=[reward_fn], args=grpo_config, train_dataset=train_dataset)
if 'processing_class' in _sig:
    trainer = GRPOTrainer(processing_class=tokenizer, **_common)
else:
    trainer = GRPOTrainer(tokenizer=tokenizer, **_common)

trainer.model.generation_config = GenerationConfig(
    max_new_tokens     = MAX_NEW_TOKENS,
    temperature        = TEMPERATURE,
    top_p              = TOP_P,
    top_k              = TOP_K,
    do_sample          = True,
    repetition_penalty = REPETITION_PENALTY,
    pad_token_id       = tokenizer.eos_token_id,
    eos_token_id       = tokenizer.eos_token_id,
)

def _has_checkpoint(d):
    p = Path(d)
    return p.exists() and any(c.name.startswith('checkpoint-') for c in p.iterdir())

torch.cuda.empty_cache(); gc.collect()
resume = _has_checkpoint(OUTPUT_DIR)
print(f'Training (resume={resume}) ...')
trainer.train(resume_from_checkpoint=resume)
print('Training complete')

## 13) Save adapters

Only the LoRA adapters are saved (~50 MB). Re-load with `PeftModel.from_pretrained(...)`.

In [ ]:
Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print('Saved adapters to', SAVE_DIR)

# Optional: push to HF Hub
# HF_REPO_ID = 'YOUR_HF_USERNAME/urban-planner-grpo-t4'
# model.push_to_hub(HF_REPO_ID); tokenizer.push_to_hub(HF_REPO_ID)

## 14) Plot the reward curve

Two panels: smoothed mean reward (top) and rolling std (bottom). The std panel is the **early-warning gauge**: if it ever drops near 0 the GRPO group has collapsed and learning will stall.

In [ ]:
import matplotlib.pyplot as plt

steps_l, rews_l = [], []
for entry in trainer.state.log_history:
    if 'reward' in entry:
        steps_l.append(entry['step']); rews_l.append(entry['reward'])
steps_arr = np.asarray(steps_l)
rews_arr  = np.asarray(rews_l, dtype=float)
W = max(5, len(rews_arr) // 20) if len(rews_arr) else 5

def _smooth(a, w=W):
    return a if len(a) < w else np.convolve(a, np.ones(w)/w, mode='valid')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True,
                                gridspec_kw={'height_ratios': [3, 1]})
ax1.plot(steps_arr, rews_arr, alpha=0.25, color='#6c8ebf', lw=1, label='raw')
if len(rews_arr) >= W:
    sm = _smooth(rews_arr); sm_steps = steps_arr[W-1:]
    ax1.plot(sm_steps, sm, color='#3266ad', lw=2.5, label=f'smoothed (w={W})')
    rstd = np.array([rews_arr[max(0, i-W):i].std() for i in range(W, len(rews_arr)+1)])
    ax1.fill_between(sm_steps, sm - rstd, sm + rstd, alpha=0.15, color='#3266ad', label='±1σ')
    ax2.plot(sm_steps, rstd, color='#e07070', lw=1.5)
    ax2.axhline(0.05, color='#e07070', lw=0.6, ls='--', alpha=0.6)
    ax2.set_ylabel('Reward σ'); ax2.set_xlabel('Training step')
ax1.axhline(0, color='#999', lw=0.8, ls='--')
ax1.set_ylabel('Mean reward / step')
ax1.set_title('Urban Planner GRPO (T4) — reward curve')
ax1.legend(); ax1.grid(axis='y', alpha=0.3); ax2.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'reward_curve.png', dpi=140)
plt.show()
print('Saved', PLOTS_DIR / 'reward_curve.png')

## 15) Random-vs-trained comparison

Same seed (`999`), same horizon — direct, head-to-head.

In [ ]:
def _run_random(seed=999, max_steps=40):
    rng = random.Random(seed)
    env = UrbanPlannerEnvironment()
    obs = env.reset(EpisodeConfig(seed=seed, starting_budget=STARTING_BUDGET))
    out = []
    for _ in range(max_steps):
        if obs.done: break
        a = random_action(rng); obs = env.step(a); out.append(obs.reward)
    return out

def _run_trained(seed=999, max_steps=40):
    FastLanguageModel.for_inference(model)
    env = UrbanPlannerEnvironment()
    obs = env.reset(EpisodeConfig(seed=seed, starting_budget=STARTING_BUDGET))
    out = []
    for _ in range(max_steps):
        if obs.done: break
        prompt = format_prompt(observation_to_text(obs))
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True,
                           max_length=MAX_PROMPT_LEN).to(model.device)
        with torch.no_grad():
            tok = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS,
                                 temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
                                 do_sample=True, repetition_penalty=REPETITION_PENALTY,
                                 pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(tok[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        parsed = parse_tool_call(text)
        action = ({'tool_name': parsed[0], 'arguments': parsed[1]}
                  if parsed is not None
                  else {'tool_name': 'get_city_state', 'arguments': {'region': 'all'}})
        obs = env.step(action); out.append(obs.reward)
    return out

br = np.array(_run_random(seed=999))
tr = np.array(_run_trained(seed=999))
print(f'Random  mean={br.mean():+.3f} steps={len(br)}')
print(f'Trained mean={tr.mean():+.3f} steps={len(tr)}')

W2 = max(3, min(len(br), len(tr)) // 5)
def _esmooth(a, w=W2):
    return np.array(a) if len(a) < w else np.convolve(a, np.ones(w)/w, mode='valid')

br_sm, tr_sm = _esmooth(br), _esmooth(tr)
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(br, color='#e07070', alpha=0.25, lw=1)
ax.plot(np.arange(len(br_sm)), br_sm, color='#e07070', lw=2,
        label=f'Random (mean={br.mean():+.3f})')
ax.plot(tr, color='#5a9e6f', alpha=0.25, lw=1)
ax.plot(np.arange(len(tr_sm)), tr_sm, color='#5a9e6f', lw=2.5,
        label=f'GRPO trained (mean={tr.mean():+.3f})')
ax.axhline(0, color='#999', lw=0.6, ls='--')
ax.set_xlabel('Step'); ax.set_ylabel('Reward')
ax.set_title('Urban Planner — trained vs random (seed=999)')
ax.legend(); ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig(PLOTS_DIR / 'reward_comparison.png', dpi=140)
plt.show()
print('Saved', PLOTS_DIR / 'reward_comparison.png')

## Why each hyperparameter

- **`Qwen2.5-3B-Instruct`** — 7B+QLoRA+GRPO+KV cache for 4 generations doesn't fit T4 reliably; 3B leaves ~6 GB headroom. Qwen2.5 already produces clean tool-call JSON, so RL focuses on *content* quality.
- **`MAX_SEQ_LENGTH=640`** — activations scale linearly with seq_len. 1024 → 640 cuts ~35% activation memory and speeds the forward pass. Tool-call JSON < 80 tokens, so 128 completion is plenty.
- **LoRA `r=16, α=32`** — `r=8` is too small to absorb new behavior, `r=32` triggers OOM with 4 generations. r=16 is the sweet spot.
- **`num_generations=4`** — 2 collapses to identical completions, 6+ OOMs on T4. 4 reliably yields meaningful std.
- **`per_device_batch=1, grad_accum=4`** → effective batch 16, low VRAM.
- **`beta=0.0`** — *disables* the reference-model KL term ⇒ TRL doesn't load a frozen copy ⇒ ~3.5 GB freed. Stability comes from PPO clipping (`epsilon=0.2`) + gradient norm clipping.
- **`learning_rate=5e-6`, cosine, `warmup_ratio=0.05`** — GRPO with LoRA on Qwen routinely diverges above 1e-5; 5e-6 + cosine is the most reliable setting.
- **`temperature=1.0, top_p=0.95, top_k=50`** — diverse sampling at *training* time so GRPO has gradient. We could lower it for inference if needed.
- **`max_grad_norm=1.0`** — the previous notebook used 0.05, which silently zeroed most gradients and contributed to the plateau.
- **Reward function** — the parse-fail penalty is **decisive** (-1.0) so the model learns the JSON schema in the first ~30 steps. Most variance comes from the per-completion **rubric Δ**, which differs across the GRPO group because each completion changes the city state differently.
- **Shared reward env** — rebuilding `UrbanPlannerEnvironment` (which spins up a FastMCP server) for every reward call was the silent speed killer in the previous run. Reusing one env via `reset()` is ~3× faster.